# cuML - GPU 加速的機器學習

> cuML 提供與 scikit-learn 相容的 API，讓傳統機器學習演算法在 GPU 上獲得 5-175 倍加速。

## GPU 加速系列 Notebooks

1. [Numba CUDA 基礎](./01-Introduction-to-cuda-python-with-numba.ipynb)
2. [GPU 加速概覽與環境建置](./02-GPU-Acceleration-Overview-and-Environment-Setup.ipynb)
3. [CuPy - GPU 版 NumPy](./03-CuPy-GPU-Accelerated-NumPy.ipynb)
4. [cuDF - GPU 版 pandas](./04-cuDF-GPU-Accelerated-DataFrames.ipynb)
5. **[cuML - GPU 版 scikit-learn](./05-cuML-GPU-Accelerated-Machine-Learning.ipynb) ← 目前位置**
6. [cuGraph - GPU 版 NetworkX](./06-cuGraph-GPU-Accelerated-Graph-Analytics.ipynb)
7. [Dask-CUDA 多 GPU 處理](./07-Dask-CUDA-Multi-GPU-and-Large-Scale-Processing.ipynb)
8. [RAPIDS 端到端實戰](./08-RAPIDS-End-to-End-Data-Analysis-Pipeline.ipynb)

---
## 0. 環境檢查與安裝

In [ ]:
import shutil
import subprocess

if shutil.which('nvidia-smi'):
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print(result.stdout)
    GPU_AVAILABLE = True
else:
    print('未偵測到 NVIDIA GPU。')
    print('建議使用 Google Colab (免費 T4 GPU): https://colab.research.google.com')
    GPU_AVAILABLE = False

In [ ]:
# 安裝 cuML
# !pip install cuml-cu12    # CUDA 12

In [ ]:
import numpy as np
import time
from sklearn.datasets import make_classification, make_blobs, make_regression
from sklearn.model_selection import train_test_split

if GPU_AVAILABLE:
    import cuml
    import cudf
    print(f'cuML 版本: {cuml.__version__}')
else:
    print('本 Notebook 將只示範 scikit-learn（CPU）版本。')

---
## 1. cuML vs scikit-learn API 對照

cuML 的 API 設計與 scikit-learn **高度一致**：

| scikit-learn | cuML | 演算法 |
|--------------|------|--------|
| `sklearn.linear_model.LinearRegression` | `cuml.LinearRegression` | 線性迴歸 |
| `sklearn.linear_model.LogisticRegression` | `cuml.LogisticRegression` | 邏輯迴歸 |
| `sklearn.cluster.KMeans` | `cuml.KMeans` | K-Means 分群 |
| `sklearn.ensemble.RandomForestClassifier` | `cuml.RandomForestClassifier` | 隨機森林 |
| `sklearn.decomposition.PCA` | `cuml.PCA` | 主成分分析 |
| `sklearn.neighbors.KNeighborsClassifier` | `cuml.KNeighborsClassifier` | KNN |
| `sklearn.svm.SVC` | `cuml.SVC` | 支援向量機 |
| `umap.UMAP` | `cuml.UMAP` | UMAP 降維 |
| `hdbscan.HDBSCAN` | `cuml.HDBSCAN` | HDBSCAN 分群 |

### 使用模式完全相同

```python
# scikit-learn
from sklearn.cluster import KMeans
model = KMeans(n_clusters=5)
model.fit(X)
labels = model.predict(X)

# cuML（只改 import）
from cuml import KMeans
model = KMeans(n_clusters=5)
model.fit(X)
labels = model.predict(X)
```

---
## 2. 線性迴歸 (Linear Regression)

In [ ]:
# 產生迴歸資料
N = 1_000_000  # 一百萬筆
n_features = 50

X, y = make_regression(n_samples=N, n_features=n_features, noise=0.1, random_state=42)
X = X.astype(np.float32)
y = y.astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'訓練資料: {X_train.shape[0]:,} 筆 × {n_features} 特徵')
print(f'測試資料: {X_test.shape[0]:,} 筆')

In [ ]:
from sklearn.linear_model import LinearRegression as skLinearRegression
from sklearn.metrics import r2_score

# CPU (scikit-learn)
start = time.time()
model_cpu = skLinearRegression()
model_cpu.fit(X_train, y_train)
pred_cpu = model_cpu.predict(X_test)
cpu_time = time.time() - start

r2_cpu = r2_score(y_test, pred_cpu)
print(f'sklearn LinearRegression')
print(f'  耗時: {cpu_time:.4f} 秒')
print(f'  R²:   {r2_cpu:.6f}')

In [ ]:
# GPU (cuML)
if GPU_AVAILABLE:
    from cuml import LinearRegression as cuLinearRegression

    start = time.time()
    model_gpu = cuLinearRegression()
    model_gpu.fit(X_train, y_train)
    pred_gpu = model_gpu.predict(X_test)
    gpu_time = time.time() - start

    # 轉回 NumPy 計算 R²
    r2_gpu = r2_score(y_test, pred_gpu.get() if hasattr(pred_gpu, 'get') else pred_gpu)
    print(f'cuML LinearRegression')
    print(f'  耗時: {gpu_time:.4f} 秒')
    print(f'  R²:   {r2_gpu:.6f}')
    print(f'  加速: {cpu_time / gpu_time:.1f}x')
else:
    print('需要 GPU 環境。預期加速: 5-20x')

---
## 3. K-Means 分群 (Clustering)

In [ ]:
# 產生分群資料
N_cluster = 500_000
n_clusters = 10
n_features_cluster = 20

X_cluster, y_true = make_blobs(
    n_samples=N_cluster,
    n_features=n_features_cluster,
    centers=n_clusters,
    random_state=42
)
X_cluster = X_cluster.astype(np.float32)
print(f'分群資料: {N_cluster:,} 筆 × {n_features_cluster} 特徵, {n_clusters} 群')

In [ ]:
from sklearn.cluster import KMeans as skKMeans

# CPU (scikit-learn)
start = time.time()
kmeans_cpu = skKMeans(n_clusters=n_clusters, max_iter=300, n_init=1, random_state=42)
labels_cpu = kmeans_cpu.fit_predict(X_cluster)
cpu_time = time.time() - start

print(f'sklearn KMeans ({N_cluster:,} 筆, {n_clusters} 群)')
print(f'  耗時:   {cpu_time:.4f} 秒')
print(f'  慣量:   {kmeans_cpu.inertia_:.0f}')

In [ ]:
# GPU (cuML)
if GPU_AVAILABLE:
    from cuml import KMeans as cuKMeans

    start = time.time()
    kmeans_gpu = cuKMeans(n_clusters=n_clusters, max_iter=300, n_init=1, random_state=42)
    labels_gpu = kmeans_gpu.fit_predict(X_cluster)
    gpu_time = time.time() - start

    print(f'cuML KMeans ({N_cluster:,} 筆, {n_clusters} 群)')
    print(f'  耗時:   {gpu_time:.4f} 秒')
    print(f'  慣量:   {kmeans_gpu.inertia_:.0f}')
    print(f'  加速:   {cpu_time / gpu_time:.1f}x')
else:
    print('需要 GPU 環境。預期加速: 30-100x')

---
## 4. 隨機森林分類 (Random Forest)

In [ ]:
# 產生分類資料
N_clf = 200_000
n_features_clf = 30

X_clf, y_clf = make_classification(
    n_samples=N_clf,
    n_features=n_features_clf,
    n_informative=15,
    n_classes=5,
    random_state=42
)
X_clf = X_clf.astype(np.float32)
y_clf = y_clf.astype(np.int32)

X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42
)
print(f'分類資料: {N_clf:,} 筆, {n_features_clf} 特徵, 5 類')

In [ ]:
from sklearn.ensemble import RandomForestClassifier as skRF
from sklearn.metrics import accuracy_score

# CPU (scikit-learn)
start = time.time()
rf_cpu = skRF(n_estimators=100, max_depth=16, random_state=42, n_jobs=-1)
rf_cpu.fit(X_train_clf, y_train_clf)
pred_cpu_rf = rf_cpu.predict(X_test_clf)
cpu_time = time.time() - start

acc_cpu = accuracy_score(y_test_clf, pred_cpu_rf)
print(f'sklearn RandomForest (100 trees)')
print(f'  耗時:   {cpu_time:.4f} 秒')
print(f'  準確率: {acc_cpu:.4f}')

In [ ]:
# GPU (cuML)
if GPU_AVAILABLE:
    from cuml.ensemble import RandomForestClassifier as cuRF

    start = time.time()
    rf_gpu = cuRF(n_estimators=100, max_depth=16, random_state=42)
    rf_gpu.fit(X_train_clf, y_train_clf)
    pred_gpu_rf = rf_gpu.predict(X_test_clf)
    gpu_time = time.time() - start

    pred_np = pred_gpu_rf.get() if hasattr(pred_gpu_rf, 'get') else np.array(pred_gpu_rf)
    acc_gpu = accuracy_score(y_test_clf, pred_np)
    print(f'cuML RandomForest (100 trees)')
    print(f'  耗時:   {gpu_time:.4f} 秒')
    print(f'  準確率: {acc_gpu:.4f}')
    print(f'  加速:   {cpu_time / gpu_time:.1f}x')
else:
    print('需要 GPU 環境。預期加速: 10-40x')

---
## 5. PCA 主成分分析

In [ ]:
# 高維資料降維
N_pca = 500_000
n_features_pca = 100
n_components = 10

X_pca = np.random.rand(N_pca, n_features_pca).astype(np.float32)
print(f'PCA 資料: {N_pca:,} 筆 × {n_features_pca} 特徵 → {n_components} 主成分')

In [ ]:
from sklearn.decomposition import PCA as skPCA

# CPU
start = time.time()
pca_cpu = skPCA(n_components=n_components)
X_pca_cpu = pca_cpu.fit_transform(X_pca)
cpu_time = time.time() - start

print(f'sklearn PCA')
print(f'  耗時: {cpu_time:.4f} 秒')
print(f'  解釋變異量: {pca_cpu.explained_variance_ratio_.sum():.4f}')

# GPU
if GPU_AVAILABLE:
    from cuml import PCA as cuPCA

    start = time.time()
    pca_gpu = cuPCA(n_components=n_components)
    X_pca_gpu = pca_gpu.fit_transform(X_pca)
    gpu_time = time.time() - start

    print(f'\ncuML PCA')
    print(f'  耗時: {gpu_time:.4f} 秒')
    print(f'  解釋變異量: {pca_gpu.explained_variance_ratio_.sum():.4f}')
    print(f'  加速: {cpu_time / gpu_time:.1f}x')

---
## 6. UMAP 降維視覺化

UMAP 是 cuML 的招牌演算法之一，GPU 加速效果極為顯著。

> UMAP (Uniform Manifold Approximation and Projection) 是一種非線性降維方法，
> 常用於高維資料的 2D/3D 視覺化，效果通常優於 t-SNE 且速度更快。

In [ ]:
# 產生有明顯群聚結構的資料
N_umap = 100_000
X_umap, y_umap = make_blobs(
    n_samples=N_umap,
    n_features=50,
    centers=8,
    random_state=42
)
X_umap = X_umap.astype(np.float32)
print(f'UMAP 資料: {N_umap:,} 筆 × 50 特徵')

In [ ]:
# CPU UMAP (需安裝 umap-learn)
try:
    from umap import UMAP as cpuUMAP

    # 為了節省時間，CPU 版只用一部分資料
    N_cpu_subset = min(N_umap, 20_000)
    start = time.time()
    umap_cpu = cpuUMAP(n_components=2, random_state=42)
    embedding_cpu = umap_cpu.fit_transform(X_umap[:N_cpu_subset])
    cpu_time = time.time() - start
    print(f'CPU UMAP ({N_cpu_subset:,} 筆): {cpu_time:.4f} 秒')

    # 推算全資料量的 CPU 時間
    estimated_full = cpu_time * (N_umap / N_cpu_subset) ** 1.5
    print(f'推算 {N_umap:,} 筆 CPU 耗時: ~{estimated_full:.0f} 秒')
except ImportError:
    print('umap-learn 未安裝 (!pip install umap-learn)')
    cpu_time = None

In [ ]:
# GPU UMAP
if GPU_AVAILABLE:
    from cuml import UMAP as cuUMAP

    start = time.time()
    umap_gpu = cuUMAP(n_components=2, random_state=42)
    embedding_gpu = umap_gpu.fit_transform(X_umap)  # 全部 100K 筆
    gpu_time = time.time() - start

    print(f'GPU UMAP ({N_umap:,} 筆): {gpu_time:.4f} 秒')
    if cpu_time is not None:
        print(f'相對 CPU 子集 ({N_cpu_subset:,} 筆) 加速: GPU 處理 {N_umap/N_cpu_subset:.0f}x 資料量卻更快')
else:
    print('需要 GPU 環境。UMAP 是 cuML 加速最顯著的演算法之一 (50-200x)')

---
## 7. Zero-Code-Change 加速 scikit-learn

RAPIDS 25.02+ 推出 **`cuml.accel`**，讓既有的 scikit-learn 程式碼自動用 GPU 加速。

### 使用方式

```python
# 在 import sklearn 之前執行
import cuml.accel
cuml.accel.install()

# 之後正常使用 scikit-learn
from sklearn.cluster import KMeans  # 自動用 GPU
from sklearn.decomposition import PCA  # 自動用 GPU
from sklearn.ensemble import RandomForestClassifier  # 自動用 GPU
```

### 運作原理

```
你的程式碼: from sklearn.cluster import KMeans
                     ↓
cuml.accel 攔截: 替換為 cuML 的 KMeans
                     ↓
        ┌─ cuML 支援的演算法 → GPU 執行
        └─ 不支援的演算法 → 回退到 sklearn (CPU)
```

### 支援的演算法（持續增加中）

- KMeans, DBSCAN, HDBSCAN
- PCA, TruncatedSVD
- LinearRegression, LogisticRegression, Ridge, Lasso, ElasticNet
- RandomForest (Classifier/Regressor)
- KNeighbors (Classifier/Regressor)
- UMAP, t-SNE

> **注意：** `cuml.accel.install()` 必須在 `import sklearn` **之前** 執行。

---
## 8. 完整 ML Pipeline：cuDF + cuML

結合 cuDF（資料前處理）和 cuML（模型訓練），全程 GPU 加速。

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler as skScaler

# ===== 產生合成資料 =====
N_pipe = 500_000
np.random.seed(42)

data = {
    'age': np.random.randint(18, 70, N_pipe).astype(np.float32),
    'income': np.random.lognormal(10, 1, N_pipe).astype(np.float32),
    'spending_score': np.random.randint(1, 100, N_pipe).astype(np.float32),
    'visit_frequency': np.random.poisson(5, N_pipe).astype(np.float32),
    'account_age_days': np.random.randint(1, 3650, N_pipe).astype(np.float32),
}
# 建立標籤：高價值客戶
high_value = ((data['income'] > 30000) & (data['spending_score'] > 50)).astype(np.int32)

print(f'Pipeline 資料: {N_pipe:,} 筆, {len(data)} 特徵')
print(f'正樣本比例: {high_value.mean():.2%}')

In [ ]:
# ===== CPU Pipeline (pandas + sklearn) =====
start_total = time.time()

# 1. 資料前處理
pdf_pipe = pd.DataFrame(data)
pdf_pipe['income_log'] = np.log1p(pdf_pipe['income'])
pdf_pipe['value_ratio'] = pdf_pipe['spending_score'] / (pdf_pipe['age'] + 1)

features = ['age', 'income_log', 'spending_score', 'visit_frequency',
            'account_age_days', 'value_ratio']
X_cpu_pipe = pdf_pipe[features].values

# 2. 標準化
scaler_cpu = skScaler()
X_cpu_scaled = scaler_cpu.fit_transform(X_cpu_pipe)

# 3. 切分
X_tr, X_te, y_tr, y_te = train_test_split(
    X_cpu_scaled, high_value, test_size=0.2, random_state=42
)

# 4. 訓練
from sklearn.ensemble import RandomForestClassifier
rf_pipe_cpu = RandomForestClassifier(n_estimators=100, max_depth=12, n_jobs=-1, random_state=42)
rf_pipe_cpu.fit(X_tr, y_tr)

# 5. 預測
pred_pipe_cpu = rf_pipe_cpu.predict(X_te)
acc_pipe_cpu = accuracy_score(y_te, pred_pipe_cpu)

cpu_total = time.time() - start_total
print(f'CPU Pipeline (pandas + sklearn)')
print(f'  總耗時: {cpu_total:.4f} 秒')
print(f'  準確率: {acc_pipe_cpu:.4f}')

In [ ]:
# ===== GPU Pipeline (cuDF + cuML) =====
if GPU_AVAILABLE:
    from cuml.preprocessing import StandardScaler as cuScaler
    from cuml.ensemble import RandomForestClassifier as cuRF
    from cuml.model_selection import train_test_split as cu_train_test_split

    start_total = time.time()

    # 1. 資料前處理 (cuDF)
    gdf_pipe = cudf.DataFrame(data)
    gdf_pipe['income_log'] = gdf_pipe['income'].log1p()
    gdf_pipe['value_ratio'] = gdf_pipe['spending_score'] / (gdf_pipe['age'] + 1)

    X_gpu_pipe = gdf_pipe[features]
    y_gpu_pipe = cudf.Series(high_value)

    # 2. 標準化 (cuML)
    scaler_gpu = cuScaler()
    X_gpu_scaled = scaler_gpu.fit_transform(X_gpu_pipe)

    # 3. 切分 (cuML)
    X_tr_g, X_te_g, y_tr_g, y_te_g = cu_train_test_split(
        X_gpu_scaled, y_gpu_pipe, test_size=0.2, random_state=42
    )

    # 4. 訓練 (cuML)
    rf_pipe_gpu = cuRF(n_estimators=100, max_depth=12, random_state=42)
    rf_pipe_gpu.fit(X_tr_g, y_tr_g)

    # 5. 預測
    pred_pipe_gpu = rf_pipe_gpu.predict(X_te_g)

    gpu_total = time.time() - start_total

    pred_np = pred_pipe_gpu.to_numpy() if hasattr(pred_pipe_gpu, 'to_numpy') else np.array(pred_pipe_gpu)
    y_te_np = y_te_g.to_numpy() if hasattr(y_te_g, 'to_numpy') else np.array(y_te_g)
    acc_pipe_gpu = accuracy_score(y_te_np, pred_np)

    print(f'GPU Pipeline (cuDF + cuML)')
    print(f'  總耗時: {gpu_total:.4f} 秒')
    print(f'  準確率: {acc_pipe_gpu:.4f}')
    print(f'  加速:   {cpu_total / gpu_total:.1f}x')
else:
    print('需要 GPU 環境。預期全流程加速: 10-30x')

---
## 9. 效能總結

| 演算法 | 典型加速倍數 | 資料量建議 |
|--------|-------------|------------|
| Linear Regression | 5-20x | > 10 萬筆 |
| Logistic Regression | 5-15x | > 10 萬筆 |
| K-Means | 30-100x | > 5 萬筆 |
| Random Forest | 10-40x | > 5 萬筆 |
| PCA | 20-80x | > 10 萬筆 |
| UMAP | 50-200x | > 1 萬筆 |
| HDBSCAN | 30-150x | > 5 萬筆 |
| KNN | 10-50x | > 10 萬筆 |

### 最佳實踐

1. **全程 GPU**：cuDF 前處理 → cuML 訓練 → cuML 預測，避免中間轉換
2. **使用 float32**：GPU 的 float32 運算速度遠快於 float64
3. **批次處理**：一次傳入大量資料比分批傳入更有效率
4. **超參數調校**：GPU 加速讓 GridSearchCV / RandomizedSearchCV 變得可行
5. **新專案用 `cuml.accel`**：零成本獲得 GPU 加速

---
## 參考資源

- [cuML 官方文件](https://docs.rapids.ai/api/cuml/stable/)
- [cuML 支援的演算法列表](https://docs.rapids.ai/api/cuml/stable/api/)
- [cuml.accel 零程式碼變更加速](https://docs.rapids.ai/api/cuml/stable/cuml_accel/)
- [RAPIDS + XGBoost 整合](https://xgboost.readthedocs.io/en/latest/gpu/index.html)

> **下一步：** 前往 [06-cuGraph-GPU-Accelerated-Graph-Analytics.ipynb](./06-cuGraph-GPU-Accelerated-Graph-Analytics.ipynb) 學習 GPU 加速的圖形分析。